In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.impute import SimpleImputer
import os
from sklearn.svm import LinearSVC

# -------------------------- 1. 文件路径处理 --------------------------
data_path = "D:\\notebook文件\\实验2\\weatherAUS.csv"  # 双反斜杠（Windows专用）
# data_path = "D:/notebook文件/实验2/weatherAUS.csv"  # 正斜杠（跨平台）
# data_path = r"D:\notebook文件\实验2\weatherAUS.csv"  # 原始字符串（推荐）

# 路径容错
if not os.path.exists(data_path):
    print(f"警告：文件路径不存在！当前路径：{data_path}")
    current_dir = os.getcwd()
    csv_files = [f for f in os.listdir(current_dir) if f.endswith(".csv")]
    if csv_files:
        print(f"\n当前目录下找到CSV文件：{csv_files}")
        data_path = os.path.join(current_dir, csv_files[0])
        print(f"已自动使用文件：{data_path}")
    else:
        print("\n当前目录下未找到CSV文件，请手动确认文件位置！")

# -------------------------- 2. 数据加载 --------------------------
try:
    df = pd.read_csv(data_path)
    print(f"成功加载数据，数据集形状：{df.shape}")
except FileNotFoundError:
    print("本地文件未找到，尝试从Kaggle下载...")
    try:
        os.system("kaggle datasets download -d jsphyg/weather-dataset-rattle-package -o ./ --unzip")
        df = pd.read_csv("weatherAUS.csv")
        print("从Kaggle下载并加载数据成功！")
    except Exception as e:
        print(f"Kaggle下载失败：{e}")
        print("请手动从 https://www.kaggle.com/jsphyg/weather-dataset-rattle-package 下载数据集")
        exit()

# -------------------------- 3. 数据预处理（核心修复：处理缺失值+编码） --------------------------
# 筛选关键特征
features = [
    "Date", "Location", "MinTemp", "MaxTemp", "Rainfall", "Evaporation",
    "Sunshine", "WindGustDir", "WindGustSpeed", "WindDir9am", "WindDir3pm",
    "WindSpeed9am", "WindSpeed3pm", "Humidity9am", "Humidity3pm",
    "Pressure9am", "Pressure3pm", "Cloud9am", "Cloud3pm", "Temp9am",
    "Temp3pm", "RainToday", "RainTomorrow"
]
df = df[features].copy()

# 第一步：先处理RainTomorrow和RainToday的缺失值和异常值
# 1. 过滤RainTomorrow的无效值（仅保留Yes/No，删除nan和其他异常值）
df = df[df["RainTomorrow"].isin(["Yes", "No", "YES", "NO"])].copy()
# 2. 填充RainToday的缺失值（用"NO"填充，因为下雨样本较少，填充为多数类更合理）
df["RainToday"] = df["RainToday"].fillna("NO")
# 3. 过滤RainToday的无效值（仅保留Yes/No）
df = df[df["RainToday"].isin(["Yes", "No", "YES", "NO"])].copy()

# 第二步：统一大小写并编码
label_encoder = LabelEncoder()
# 统一转为大写（避免Yes/YES/yes的差异）
df["RainTomorrow"] = df["RainTomorrow"].str.upper()
df["RainToday"] = df["RainToday"].str.upper()
# 编码（此时两个字段都只有YES/NO，无nan）
df["RainTomorrow"] = label_encoder.fit_transform(df["RainTomorrow"])
df["RainToday"] = label_encoder.transform(df["RainToday"])
print("编码后的类别映射：", dict(zip(label_encoder.classes_, [0, 1])))  # 输出：{'NO':0, 'YES':1}

# 第三步：分类特征独热编码
categorical_cols = ["Location", "WindGustDir", "WindDir9am", "WindDir3pm"]
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# 第四步：日期特征处理
df["Date"] = pd.to_datetime(df["Date"])
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df.drop("Date", axis=1, inplace=True)

# 第五步：处理其他数值特征的缺失值（用中位数填充）
imputer = SimpleImputer(strategy="median")
df_imputed = pd.DataFrame(
    imputer.fit_transform(df),
    columns=df.columns
)

# -------------------------- 4. 数据集拆分 --------------------------
X = df_imputed.drop("RainTomorrow", axis=1)
y = df_imputed["RainTomorrow"].astype(int)  # 强制整数类别

# 校验目标变量（确保只有2类）
print("\n目标变量唯一类别：", np.unique(y))
print("类别数量：", len(np.unique(y)))  # 正常输出：2

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y  # 保持类别分布
)

# -------------------------- 5. 特征标准化 --------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -------------------------- 6. SVM模型训练（可选：线性核加快速度） --------------------------
# 推荐用linear核（训练快，准确率仅略降），rbf核需等待更久
svm_model = LinearSVC(C=1.0, random_state=42)
#svm_model = SVC(kernel="linear", C=1.0, random_state=42)  # 线性核
# svm_model = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)  # RBF核（慢）

print("\n开始训练SVM模型...")
svm_model.fit(X_train_scaled, y_train)
y_pred = svm_model.predict(X_test_scaled)

# -------------------------- 7. 模型评估（修复分类报告参数） --------------------------
print("\n========== 模型评估结果 ==========")
accuracy = accuracy_score(y_test, y_pred)
print(f"测试集准确率：{accuracy:.4f}")

# 分类报告（显式指定labels，避免类别不匹配）
print("\n分类报告：")
print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=["不下雨（NO）", "下雨（YES）"],
    zero_division=0  # 避免无预测样本时报错
))

# 混淆矩阵
print("\n混淆矩阵：")
conf_matrix = confusion_matrix(y_test, y_pred)
print(conf_matrix)
print(f"真阴性（实际不下雨，预测不下雨）：{conf_matrix[0][0]}")
print(f"假阳性（实际不下雨，预测下雨）：{conf_matrix[0][1]}")
print(f"假阴性（实际下雨，预测不下雨）：{conf_matrix[1][0]}")
print(f"真阳性（实际下雨，预测下雨）：{conf_matrix[1][1]}")


成功加载数据，数据集形状：(145460, 23)
编码后的类别映射： {'NO': 0, 'YES': 1}

目标变量唯一类别： [0 1]
类别数量： 2

开始训练SVM模型...

========== 模型评估结果 ==========
测试集准确率：0.8493

分类报告：
              precision    recall  f1-score   support

     不下雨（NO）       0.87      0.95      0.91     33095
     下雨（YES）       0.75      0.49      0.60      9563

    accuracy                           0.85     42658
   macro avg       0.81      0.72      0.75     42658
weighted avg       0.84      0.85      0.84     42658


混淆矩阵：
[[31498  1597]
 [ 4831  4732]]
真阴性（实际不下雨，预测不下雨）：31498
假阳性（实际不下雨，预测下雨）：1597
假阴性（实际下雨，预测不下雨）：4831
真阳性（实际下雨，预测下雨）：4732
